## Detecting collisions

Para detectar una colisión:
``` python
contact_points = p.getContactPoints(bodyA=self.DRONE_IDS[0])
```
Contact points is a tuple:
- contactFlag 
- bodyUniqueIdA 
- bodyUniqueIdB 
- linkIndexA 
- linkIndexB 
- positionOnA 
- positionOnB 
- contactNormalOnB 
- contactDistance 
- normalForce 
- lateralFriction1 
- lateralFrictionDir1 
- lateralFriction2 
- lateralFrictionDir2 



## Vector field

**Vector field formula following [Piniés and Tardós](https://ieeexplore.ieee.org/document/1642314)**

$$
H = \frac{1}{4\pi||p||^5}Am \\
$$
Where
$$
A = \begin{bmatrix} 
2x_p^2 - y_p^2 - z_p^2 & 3x_py_p & 3x_pz_p \\
3x_py_p & 2y_p^2 - x_p^2 - z_p^2 & 3y_pz_p \\
3x_pz_p & 3y_pz_p & 2z_p^2 - x_p^2 - y_p^2 
\end{bmatrix}
$$
$p = p_r - p_t$

$p_r = \text{position of the receiver}$

$p_t = \text{position of the transmitter}$

$m = \text{magnetic momentum, in this case} = R_t = [\psi, \theta, \phi] (\text{rotation of the transmitter, yaw, pitch roll})$

In [ ]:
from itertools import permutations

import numpy as np
import plotly.express as px

E1 = np.array([[1, 0, 0]]).T

def transformation_matrix_3d(vector):
    matrix = np.zeros((3,3))
    x, y, z = vector[0], vector[1], vector[2],
    x2, y2, z2 = x**2, y**2, z**2
    matrix[0,0] = 2 * x2 - y2 - z2
    matrix[1,1] = 2 * y2 - x2 - z2
    matrix[2,2] = 2 * z2 - y2 - x2
    matrix[[0,1],[1,0]] = 3 * x * y
    matrix[[0,2],[2,0]] = 3 * x * z
    matrix[[1,2],[2,1]] = 3 * y * z
    return matrix

def transformation_matrix_2d(vector):
    matrix = np.zeros((2,2))
    x, y = vector[0], vector[1]
    x2, y2 = x**2, y**2
    matrix[0,0] = 2 * x2 - y2
    matrix[1,1] = 2 * y2 - x2
    matrix[[0,1],[1,0]] = 3 * x * y
    return matrix

def vector_field(t_pos, r_pos, t_rot):
    r_t_vector = r_pos - t_pos
    numerator = transformation_matrix_3d(r_t_vector) @ t_rot
    denominator = 4 * np.pi * np.linalg.norm(r_t_vector)**5
    result = numerator / denominator
    result = result / np.linalg.norm(result)
    return result

t_pos = np.array([0,0,0])
r_pos = np.array([.5,.5,.3])
t_rot = np.array([0.3,.2,.1])



vector_field(t_pos ,r_pos, t_rot)


### Visualisation

In [ ]:
import pandas as pd

x = np.linspace(-9,9,20)
y = np.linspace(-9,9,20)
z = np.linspace(-9,9,20)
xv, yv, zv = np.meshgrid(x, y, z)

start_positions = np.vstack([xv.flat, yv.flat, zv.flat]).T
def transformation_matrix_4d(vector):
    matrix = np.zeros((vector.shape[0], 3, 3))
    x, y, z = vector[:, 0], vector[:, 1], vector[:, 2],
    x2, y2, z2 = x**2, y**2, z**2
    matrix[:,0,0] = 2 * x2 - y2 - z2
    matrix[:,1,1] = 2 * y2 - x2 - z2
    matrix[:,2,2] = 2 * z2 - y2 - x2
    matrix[:, 0, 1] = matrix[:, 1, 0] = 3 * x * y
    matrix[:, 0, 2] = matrix[:, 2, 0] = 3 * x * z
    matrix[:, 1, 2] = matrix[:, 2, 1] = 3 * y * z
    return matrix

def vector_field_batch(t_pos, r_pos, t_rot):
    r_t_vector = r_pos - t_pos
    numerator = transformation_matrix_4d(r_t_vector) @ t_rot
    denominator = 4 * np.pi * np.linalg.norm(r_t_vector)**5
    result = numerator / denominator
    result = result / np.linalg.norm(result)
    return result


end_positions = start_positions + vector_field_batch(t_pos, start_positions, t_rot)*60
start = pd.DataFrame(start_positions, columns=['x', 'y', 'z'])
start['start'] = 0
end = pd.DataFrame(end_positions, columns=['x', 'y', 'z'])
end['start'] = 1

positions = pd.concat([start, end]).reset_index()

In [ ]:
transformation_matrix_4d(start_positions)

In [ ]:
import plotly.graph_objects as go

frames = []
for i in range(20):
    mask = start_positions[:,2] == start_positions[i, 2]
    s = start_positions[mask,:-1]
    e = end_positions[mask,:-1]
    frame = go.Frame(data=[
                        go.Scatter(x=[s[j,0], e[j,0]],
                                    y=[s[j,1], e[j,1]], 
                                    marker=dict(color='red'),
                                    mode='lines')
                    for j in range(s.shape[0])])
    frames.append(frame)

mask = start_positions[:,2] == start_positions[0, 2]
s = start_positions[mask,:-1]
e = end_positions[mask,:-1]
init = [go.Scatter(x=[s[j,0], e[j,0]],
                                    y=[s[j,1], e[j,1]], 
                                    marker=dict(color='red'),
                                    mode='lines')
                    for j in range(s.shape[0])]

fig = go.Figure(
    data=init,
    frames=frames
)
fig.update_layout(width=900, height=900, showlegend=False,
                  xaxis=dict(autorange=False, range=[-10,10]),
                  yaxis=dict(autorange=False, range=[-10,10]),
                  updatemenus=[dict(
            type="buttons",
            buttons=[dict(label="Play",
                          method="animate",
                          args=[None])])])

In [ ]:
init = [go.Scatter3d(x=[start_positions[j,0], end_positions[j,0]],
                     y=[start_positions[j,1], end_positions[j,1]], 
                     z=[start_positions[j,2], end_positions[j,2]], 
                     marker=dict(color='red'),
                     mode='lines')
                    for j in range(start_positions.shape[0])]

fig = go.Figure(
    data=init,
)
fig.update_layout(width=900, height=900, showlegend=False,)

## Signal intensity
Following [Azzollini et al.](https://arxiv.org/abs/2106.14514)

$$
||h|| = \frac{1}{4\pi||p||^3}\sqrt{1+3\frac{p^TMp}{||p||^2}}
$$
Where
$M = R_t^Te1e1^TR_t$

$e1=[1, 0, 0]^T$

In [ ]:
rp = np.array([[0,0,1], [1,2,3]], dtype=np.float32)

def signal_intensity(t_pos, r_pos, t_rot, scaler=1):
    E1 = np.array([[1,0,0]]).T
    M = t_rot.T @ E1 * E1.T @ t_rot
    r_t_vector = r_pos - t_pos
    r_t_norm = np.linalg.norm(r_t_vector, axis=1)
    inside_sqrt = 1 + (3 * np.diag(r_t_vector * M @ r_t_vector.T)) / r_t_norm**2
    intensity = (scaler * inside_sqrt ** .5) / (4 * np.pi * r_t_norm**3)
    return intensity

signal_intensity(t_pos, rp, t_rot)

In [ ]:
rp[:,[0,1]] = np.random.uniform(-5, 5,(rp.shape[0], 2))
rp

In [ ]:


def apply_intensity(row):
    r_pos = row.iloc[[0,1,2]].values
    return signal_intensity(t_pos, r_pos, np.array([.5,0.5,0.5]), 100)

start['intensity'] = start.apply(apply_intensity, axis=1)

selection = start.loc[start['z'] == start['z'].unique()[10], ['x', 'y', 'intensity']]
grid = selection.reset_index().groupby(['x', 'y'])['intensity'].aggregate('first').unstack()
px.imshow(grid, width=500)

In [ ]:
start['z'].unique()

# LoRa-SaR-dataset

In [ ]:
import plotly.express as px
import pandas as pd
dron_df  = pd.read_csv('data/LoRa-SaR-dataset/dataset/drone/drone.csv')

selection = dron_df.loc[dron_df['depth'] == 0.5, ['x', 'y', 'rssi']]
grid = selection.reset_index().groupby(['x', 'y'])['rssi'].aggregate('first').unstack()
px.imshow(grid, width=500)


In [ ]:
cross = pd.read_csv('data/LoRa-SaR-dataset/dataset/cross/march/cross.csv')

def get_x(row):
    if row['rx_pos'] == 'south':
        return -row['distance']
    if row['rx_pos'] == 'north':
        return row['distance']
    return 0

def get_y(row):
    if row['rx_pos'] == 'west':
        return row['distance']
    if row['rx_pos'] == 'east':
        return -row['distance']
    return 0

cross['x'] = cross.apply(get_x, axis=1)
cross['y'] = cross.apply(get_y, axis=1)

In [ ]:
grid_cross = cross.loc[cross['depth']==1] \
                .reset_index() \
                .groupby(['x', 'y'])['rssi'] \
                .aggregate('first') \
                .unstack() \
                .fillna(
                    cross['rssi'].min()-5
                )
px.imshow(grid_cross, width=500)
